# Improved 5-Class Ensemble (Progressive K=5 and Beyond)

This notebook trains a **5-class** oral cavity ensemble using the same **progressive_K5** strategy that won the binary freeze-strategy comparison (head warm-up 5 epochs → unfreeze last 5 backbone blocks). Additional strategies (e.g. head_only, full_finetune) may be added later for comparison.

## Goals
1. **Progressive_K5 for 5-class** – Train 3-model ensemble with progressive unfreezing on the 5-class dataset only (`oral_cavity_5class_dataset`).
2. **Compare to current 5-class production** – Evaluate on T5 (and optionally val) to compare with the current full-finetune 5-class ensemble.
3. **Agreement analysis** – Compare binary progressive_K5 ensemble with this 5-class progressive_K5 ensemble (or current 5-class) on frozen sets (T5/TM) to document referral vs label agreement.

## Data
- **Source:** 5-class dataset only (`train/`, `val/`, `test/` with classes Benign, Healthy, OCA, OPMD, Unknown).
- **Evaluation:** T5 (5-class test set) and optionally frozen test sets via `frozen_test_sets` + `evaluate_on_frozen_sets.py`.

## 1. Configuration

In [1]:
import os
from pathlib import Path
from datetime import datetime
import json

DATA_DIR_5CLASS = os.path.expanduser("~/dev/oral_cavity_5class_dataset")
NOTEBOOK_DIR = Path(".").resolve()
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = os.environ.get("EXPERIMENT_NAME") or f"improved_5class_{timestamp}"

CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "num_epochs": 50,
    "patience": 12,
    "num_workers": 0,
    "seed": 42,
    "augmentation": "medical",
    "label_smoothing": 0.05,
    "dropout_rates": [0.3, 0.1, 0.2],
    "ensemble_weights": [0.30, 0.40, 0.30],
    "class_names": ["Benign", "Healthy", "OCA", "OPMD", "Unknown"],
    "num_classes": 5,
    "progressive_K5": {
        "lr_head": 1e-3,
        "lr_backbone": 1e-4,
        "warmup_epochs": 5,
        "unfreeze_last_k": 5,
    },
}

OUTPUT_DIR = NOTEBOOK_DIR / f"results/improved_5class/{run_name}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
TB_LOG_DIR = OUTPUT_DIR / "tb_logs"
TB_LOG_DIR.mkdir(exist_ok=True)

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({**CONFIG, "run_name": run_name, "image_size": list(CONFIG["image_size"])}, f, indent=2, default=str)

print("Improved 5-Class Ensemble (progressive_K5)")
print("=" * 60)
print(f"  Data: {DATA_DIR_5CLASS}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Epochs: {CONFIG['num_epochs']} (+ {CONFIG['progressive_K5']['warmup_epochs']} warmup)")
print(f"  Patience: {CONFIG['patience']}")

Improved 5-Class Ensemble (progressive_K5)
  Data: /Users/alokdwivedi/dev/oral_cavity_5class_dataset
  Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_5class/improved_5class_20260209_172821
  Epochs: 50 (+ 5 warmup)
  Patience: 12


## 2. Imports and Device

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("Warning: albumentations not installed. Using basic augmentation.")

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


## 3. Load 5-Class Data (train / val)

In [3]:
from torchvision import datasets

def get_5class_loaders(data_dir, config):
    """Build train/val loaders from 5-class dataset (train/ and val/ folders with class subdirs)."""
    train_dir = Path(data_dir) / "train"
    val_dir = Path(data_dir) / "val"
    if not train_dir.exists() or not val_dir.exists():
        raise FileNotFoundError(f"Expected {train_dir} and {val_dir} with class subdirs (Benign, Healthy, OCA, OPMD, Unknown)")

    # We will use ImageFolder-style: need same class order everywhere
    class_names = config["class_names"]
    size = config["image_size"]
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize(size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    train_ds = datasets.ImageFolder(str(train_dir), transform=train_tf)
    val_ds = datasets.ImageFolder(str(val_dir), transform=val_tf)
    # Ensure class order matches config (ImageFolder uses sorted folder names)
    assert train_ds.classes == val_ds.classes, "Train and val class order must match"
    print(f"Classes: {train_ds.classes}")
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

    train_loader = DataLoader(
        train_ds, batch_size=config["batch_size"], shuffle=True,
        num_workers=config["num_workers"], pin_memory=False,
    )
    val_loader = DataLoader(
        val_ds, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"],
    )
    return train_loader, val_loader, train_ds.classes

In [4]:
train_loader, val_loader, CLASS_NAMES = get_5class_loaders(DATA_DIR_5CLASS, CONFIG)
NUM_CLASSES = len(CLASS_NAMES)

Classes: ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']
Train: 2757, Val: 531


## 4. Model Architecture (5-class, freeze/unfreeze)

In [5]:
class ConservativeMobileNetV3(nn.Module):
    """MobileNetV3Large with freeze/unfreeze for progressive strategies. num_classes=5 for 5-class."""

    def __init__(self, num_classes=5, dropout_rate=0.2):
        super().__init__()
        try:
            self.backbone = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        except AttributeError:
            self.backbone = models.mobilenet_v3_large(pretrained=True)
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(960, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x

    def freeze_backbone(self):
        for p in self.backbone.features.parameters():
            p.requires_grad = False

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True

    def unfreeze_last_k_blocks(self, k):
        children = list(self.backbone.features.children())
        for layer in children[-k:]:
            for p in layer.parameters():
                p.requires_grad = True

    def get_param_groups(self, lr_head, lr_backbone=None):
        groups = [{"params": list(self.classifier.parameters()), "lr": lr_head}]
        backbone_trainable = [p for p in self.backbone.features.parameters() if p.requires_grad]
        if backbone_trainable and lr_backbone is not None:
            groups.append({"params": backbone_trainable, "lr": lr_backbone})
        return groups

    def trainable_param_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    def total_param_count(self):
        return sum(p.numel() for p in self.parameters())


class EnsembleModel1(ConservativeMobileNetV3):
    def __init__(self, num_classes=5):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][0])

class EnsembleModel2(ConservativeMobileNetV3):
    def __init__(self, num_classes=5):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][1])

class EnsembleModel3(ConservativeMobileNetV3):
    def __init__(self, num_classes=5):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][2])
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Dropout(0.2),
            nn.Linear(960, 512), nn.ReLU(inplace=True), nn.Dropout(0.1),
            nn.Linear(512, num_classes),
        )

ENSEMBLE_CLASSES = [EnsembleModel1, EnsembleModel2, EnsembleModel3]
print("5-class ensemble: 3 models, dropout=", CONFIG["dropout_rates"])

5-class ensemble: 3 models, dropout= [0.3, 0.1, 0.2]


In [ ]:
# (duplicate cell – safe to delete; evaluation code is in section 8 below)

## 5. Class Weights (5-class)

In [ ]:
# (duplicate cell – safe to delete; agreement analysis code is in section 9 below)

In [6]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = [train_loader.dataset[i][1] for i in range(len(train_loader.dataset))]
classes_ar = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=classes_ar, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32)
# Optional: boost OCA/OPMD (indices 2, 3) if desired
for i, name in enumerate(CLASS_NAMES):
    if name in ("OCA", "OPMD"):
        class_weights[i] *= 1.2
print("Class weights:", dict(zip(CLASS_NAMES, class_weights.tolist())))

Class weights: {'Benign': 0.8405487537384033, 'Healthy': 1.081176519393921, 'OCA': 3.063333511352539, 'OPMD': 0.6786462068557739, 'Unknown': 1.378499984741211}


## 6. Training: Progressive K=5 (multiclass)

In [7]:
def _make_criterion(class_weights, device, label_smoothing):
    return nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None,
        label_smoothing=label_smoothing,
    )

def _validate_5class(model, val_loader, criterion, device):
    """Validation for 5-class: accuracy and macro AUC (one-vs-rest)."""
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    all_probs, all_targets = [], []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device).long()
            out = model(data)
            loss = criterion(out, target)
            val_loss += loss.item()
            probs = torch.softmax(out, dim=1)
            _, pred = out.max(1)
            total += target.size(0)
            correct += pred.eq(target).sum().item()
            all_probs.append(probs.cpu().numpy())
            all_targets.append(target.cpu().numpy())
    acc = 100.0 * correct / total
    probs_np = np.vstack(all_probs)
    targets_np = np.concatenate(all_targets)
    try:
        auc = roc_auc_score(targets_np, probs_np, multi_class="ovr", average="macro")
    except Exception:
        auc = 0.0
    return acc, auc, val_loss / len(val_loader)

def _train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device).long()
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = out.max(1)
        total += target.size(0)
        correct += pred.eq(target).sum().item()
    return total_loss / len(train_loader), 100.0 * correct / total

def _tb_log_5class(writer, epoch, train_loss, train_acc, val_acc, val_auc, lr):
    if writer is None:
        return
    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("acc/train", train_acc, epoch)
    writer.add_scalar("acc/val", val_acc, epoch)
    writer.add_scalar("auc_macro/val", val_auc, epoch)
    writer.add_scalar("lr", lr, epoch)
    writer.flush()

In [8]:
def train_progressive_K5_5class(model_class, model_name, train_loader, val_loader, device, config, class_weights=None, tb_log_dir=None):
    """Progressive K=5 for 5-class: warmup then unfreeze last 5 blocks."""
    strat = config["progressive_K5"]
    warmup_epochs = strat["warmup_epochs"]
    main_epochs = config["num_epochs"]
    unfreeze_k = strat["unfreeze_last_k"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TENSORBOARD and tb_log_dir) else None

    model = model_class(num_classes=NUM_CLASSES).to(device)
    model.freeze_backbone()
    criterion = _make_criterion(class_weights, device, config.get("label_smoothing", 0.0))

    # Phase 1: warmup
    print(f"  [{model_name}] Phase 1 (warmup {warmup_epochs} ep): {model.trainable_param_count():,} params")
    optimizer = optim.Adam(model.classifier.parameters(), lr=strat["lr_head"])
    for epoch in range(warmup_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate_5class(model, val_loader, criterion, device)
        _tb_log_5class(writer, epoch, avg_loss, train_acc, val_acc, val_auc, strat["lr_head"])
        if epoch == warmup_epochs - 1:
            print(f"    Warmup done: val_acc={val_acc:.2f}%, val_auc_macro={val_auc:.4f}")

    # Phase 2: unfreeze last K
    model.unfreeze_last_k_blocks(unfreeze_k)
    print(f"  [{model_name}] Phase 2 (last {unfreeze_k} blocks, {main_epochs} ep): {model.trainable_param_count():,} params")
    param_groups = model.get_param_groups(lr_head=strat["lr_head"], lr_backbone=strat["lr_backbone"])
    optimizer = optim.Adam(param_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=main_epochs, eta_min=1e-6)

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0
    for epoch in range(main_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate_5class(model, val_loader, criterion, device)
        lr_now = optimizer.param_groups[0]["lr"]
        _tb_log_5class(writer, warmup_epochs + epoch, avg_loss, train_acc, val_acc, val_auc, lr_now)
        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    Ep {warmup_epochs + epoch + 1}  loss={avg_loss:.4f}  train={train_acc:.2f}%  val={val_acc:.2f}%  auc_macro={val_auc:.4f}  lr={lr_now:.2e}")
        if patience_ctr >= config["patience"]:
            print(f"    Early stop at epoch {warmup_epochs + epoch + 1}")
            break
        scheduler.step()

    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc

## 7. Run Progressive_K5 Ensemble (3 models)

In [ ]:
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

results = {}
for i, (model_class, model_name) in enumerate(zip(ENSEMBLE_CLASSES, ["ensemble_1", "ensemble_2", "ensemble_3"]), 1):
    print(f"[{i}/3] {model_name}")
    run_tb_dir = TB_LOG_DIR / "progressive_K5" / model_name
    run_tb_dir.mkdir(parents=True, exist_ok=True)
    model, best_acc, best_auc = train_progressive_K5_5class(
        model_class, model_name, train_loader, val_loader, device, CONFIG,
        class_weights=class_weights, tb_log_dir=str(run_tb_dir),
    )
    results[model_name] = {"val_acc": best_acc, "val_auc_macro": best_auc}
    torch.save(model.state_dict(), OUTPUT_DIR / "models" / f"{model_name}_progressive_K5.pth")
    print(f"  Best: val_acc={best_acc:.2f}%, val_auc_macro={best_auc:.4f}\n")

print("Results:", results)

[1/3] ensemble_1
  [ensemble_1] Phase 1 (warmup 5 ep): 247,301 params


## 8. Evaluate on T5 (5-class test) [optional]

In [ ]:
from frozen_test_sets import load_frozen_sets_v2
from torchvision import transforms
from PIL import Image

# Directory where this notebook saved the 5-class progressive_K5 checkpoints
FIVECLASS_PROGRESSIVE_K5_DIR = OUTPUT_DIR / "models"


def load_5class_ensemble(models_dir, device):
    """Load 3-model 5-class progressive_K5 ensemble from .pth checkpoints."""
    models = []
    for cls, name in zip(ENSEMBLE_CLASSES, ["ensemble_1", "ensemble_2", "ensemble_3"]):
        ckpt = models_dir / f"{name}_progressive_K5.pth"
        if not ckpt.exists():
            raise FileNotFoundError(f"Missing checkpoint: {ckpt}")
        m = cls(num_classes=NUM_CLASSES).to(device)
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        m.eval()
        models.append(m)
    return models


def evaluate_ensemble_on_T5(models_dir, device, class_names, weights=None, image_size=(224, 224)):
    """Evaluate 5-class ensemble on T5 (5-class frozen test set).

    Reports 5-class accuracy, macro one-vs-rest AUC, confusion matrix, and classification report.
    """
    if weights is None:
        weights = CONFIG["ensemble_weights"]

    models = load_5class_ensemble(models_dir, device)
    T5, TB, TC, TM, meta = load_frozen_sets_v2(FROZEN_DIR)

    label_to_idx = {name: i for i, name in enumerate(class_names)}

    tf = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    y_true, y_pred = [], []
    all_probs = []

    for row in T5:
        path = row["path"]
        label_5 = row["label_5class"]
        if label_5 not in label_to_idx:
            continue
        y_idx = label_to_idx[label_5]

        img = Image.open(path).convert("RGB")
        x = tf(img).unsqueeze(0).to(device)

        with torch.no_grad():
            logits_list = [m(x) for m in models]
            logits = sum(w * l for w, l in zip(weights, logits_list))
            probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

        y_true.append(y_idx)
        y_pred.append(int(probs.argmax()))
        all_probs.append(probs)

    if not y_true:
        print("No T5 samples evaluated (check frozen_test_sets path).")
        return None

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    all_probs = np.stack(all_probs, axis=0)

    acc = (y_pred == y_true).mean() * 100.0
    try:
        macro_auc = roc_auc_score(y_true, all_probs, multi_class="ovr", average="macro")
    except Exception:
        macro_auc = 0.0

    print(f"T5 samples used: {len(y_true)}")
    print(f"T5 accuracy: {acc:.2f}%")
    print(f"T5 macro AUC (OVR): {macro_auc:.4f}")

    print("\nClassification report (T5):")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    return {
        "acc": acc,
        "macro_auc": macro_auc,
        "cm": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": all_probs,
    }


print("Evaluating 5-class progressive_K5 ensemble on T5...")
T5_EVAL = evaluate_ensemble_on_T5(FIVECLASS_PROGRESSIVE_K5_DIR, device, CLASS_NAMES, image_size=CONFIG["image_size"])

## 8a. Export 5-Class progressive_K5 to ONNX

Convert the 3 trained 5-class `.pth` checkpoints to ONNX format so that all downstream
evaluation and agreement analysis uses **persisted ONNX models** (architecture-agnostic,
no risk of mismatch).

The ONNX models are saved alongside the `.pth` files in
`results/improved_5class/improved_5class_20260209_172821/onnx/`.

In [14]:
import onnx

# --- ONNX export helper (same as freeze_strategy_comparison.ipynb §18) --------
def convert_to_onnx(model, onnx_path, device, input_shape=(1, 3, 224, 224)):
    """Convert a PyTorch model to single-file ONNX."""
    model.eval()
    model.to(device)
    dummy = torch.randn(input_shape).to(device)
    torch.onnx.export(
        model, dummy, onnx_path,
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )
    # Re-save as single file (strip any accidental external data)
    onnx_model = onnx.load(str(onnx_path))
    onnx.save_model(onnx_model, str(onnx_path), save_as_external_data=False)
    data_file = Path(str(onnx_path) + ".data")
    if data_file.exists():
        data_file.unlink()


# --- Export the 3 ensemble models to ONNX ------------------------------------
onnx_out = OUTPUT_DIR / "onnx"
onnx_out.mkdir(parents=True, exist_ok=True)

PTH_DIR = OUTPUT_DIR / "models"

for i, (model_class, model_name) in enumerate(
    zip(ENSEMBLE_CLASSES, ["ensemble_1", "ensemble_2", "ensemble_3"]), 1
):
    pth_path = PTH_DIR / f"{model_name}_progressive_K5.pth"
    if not pth_path.exists():
        print(f"  SKIP {model_name}: {pth_path} not found")
        continue

    model = model_class(num_classes=NUM_CLASSES).to(device)
    state = torch.load(pth_path, map_location=device)
    model.load_state_dict(state)

    onnx_path = onnx_out / f"ensemble_model_{i}.onnx"
    convert_to_onnx(model, onnx_path, device)
    size_mb = onnx_path.stat().st_size / 1024 / 1024
    print(f"  ensemble_model_{i}.onnx  ({size_mb:.2f} MB)")

print(f"\n5-class ONNX models saved to: {onnx_out}")
print(f"\nUpdate model_registry.py FIVECLASS_PROGRESSIVE_K5_ONNX_DIR if the run name changes.")

W0210 10:57:52.554000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0210 10:57:52.554000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0210 10:57:52.555000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0210 10:57:52.555000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
  ensemble_model_1.onnx  (12.61 MB)


W0210 10:57:55.993000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0210 10:57:55.994000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0210 10:57:55.994000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0210 10:57:55.995000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
  ensemble_model_2.onnx  (12.61 MB)


W0210 10:57:59.492000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0210 10:57:59.493000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0210 10:57:59.493000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0210 10:57:59.494000 6173 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
  ensemble_model_3.onnx  (13.56 MB)

5-class ONNX models saved to: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_5class/improved_5class_20260209_172821/onnx

Update model_registry.py FIVECLASS_PROGRESSIVE_K5_ONNX_DIR if the run name changes.


## 9. Agreement Analysis (Binary progressive_K5 vs 5-class)

In [15]:
import onnxruntime as ort
from model_registry import (
    BINARY_PROGRESSIVE_K5_ONNX_DIR,
    FIVECLASS_PROGRESSIVE_K5_ONNX_DIR,
    ENSEMBLE_WEIGHTS as REGISTRY_WEIGHTS,
    ONNX_MODEL_NAMES,
    CANCER_CLASS_INDICES,
    FIVECLASS_CLASS_NAMES,
    get_onnx_paths,
)
from frozen_test_sets import get_tm_paths_labels_and_sources_v2

# ── Verify canonical model paths from model_registry ─────────────────────────
print("Binary  ONNX dir:", BINARY_PROGRESSIVE_K5_ONNX_DIR)
print("5-class ONNX dir:", FIVECLASS_PROGRESSIVE_K5_ONNX_DIR)
binary_onnx_paths = get_onnx_paths(BINARY_PROGRESSIVE_K5_ONNX_DIR)
fiveclass_onnx_paths = get_onnx_paths(FIVECLASS_PROGRESSIVE_K5_ONNX_DIR)
print(f"Binary models:  {[p.name for p in binary_onnx_paths]}")
print(f"5-class models: {[p.name for p in fiveclass_onnx_paths]}")


# ── ONNX inference helpers ───────────────────────────────────────────────────
def _load_onnx_sessions(onnx_dir):
    """Load 3 ONNX InferenceSessions from a directory."""
    paths = get_onnx_paths(onnx_dir)
    sessions = [ort.InferenceSession(str(p)) for p in paths]
    return sessions


def _preprocess_onnx(path, image_size=(224, 224)):
    """PIL → CHW float32 with ImageNet normalisation (matches training pipeline)."""
    from PIL import Image as PILImage
    img = PILImage.open(path).convert("RGB")
    img = img.resize(image_size)
    arr = np.array(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    arr = (arr - mean) / std
    arr = np.transpose(arr, (2, 0, 1))  # HWC → CHW
    return np.expand_dims(arr, axis=0).astype(np.float32)


def _ensemble_predict_onnx(sessions, x, weights):
    """Weighted ensemble logits from ONNX sessions."""
    input_name = sessions[0].get_inputs()[0].name
    logits_list = [sess.run(None, {input_name: x})[0] for sess in sessions]
    return sum(w * l for w, l in zip(weights, logits_list))


def _softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def _fiveclass_to_binary(class_idx, cancer_indices=CANCER_CLASS_INDICES):
    """Map a 5-class index to binary: cancer(1), non-cancer(0), or None (Unknown)."""
    name = FIVECLASS_CLASS_NAMES[class_idx]
    if name in ("Benign", "Healthy"):
        return 0
    if name in ("OCA", "OPMD"):
        return 1
    return None  # Unknown


# ── Agreement analysis on TM ─────────────────────────────────────────────────
def agreement_binary_vs_5class_onnx(
    binary_onnx_dir, fiveclass_onnx_dir, weights=None, image_size=(224, 224)
):
    """Compare binary vs 5-class progressive_K5 ONNX ensembles on TM."""
    if weights is None:
        weights = REGISTRY_WEIGHTS

    bin_sessions = _load_onnx_sessions(binary_onnx_dir)
    five_sessions = _load_onnx_sessions(fiveclass_onnx_dir)

    tm_paths, tm_labels, tm_sources = get_tm_paths_labels_and_sources_v2(FROZEN_DIR)

    bin_preds, five_preds, used_sources = [], [], []
    skipped_unknown = 0

    for i, (path, src) in enumerate(zip(tm_paths, tm_sources)):
        x = _preprocess_onnx(path, image_size)

        # Binary ensemble → 2-class logits → argmax
        bin_logits = _ensemble_predict_onnx(bin_sessions, x, weights)
        bin_pred = int(np.argmax(bin_logits, axis=1)[0])  # 0=non-cancer, 1=cancer

        # 5-class ensemble → 5-class logits → map to binary
        five_logits = _ensemble_predict_onnx(five_sessions, x, weights)
        five_class_idx = int(np.argmax(five_logits, axis=1)[0])
        five_bin = _fiveclass_to_binary(five_class_idx)

        if five_bin is None:
            skipped_unknown += 1
            continue

        bin_preds.append(bin_pred)
        five_preds.append(five_bin)
        used_sources.append(src)

        if (i + 1) % 200 == 0:
            print(f"  processed {i + 1}/{len(tm_paths)} images...")

    if not bin_preds:
        print("No TM samples used for agreement (5-class predicted Unknown for all).")
        return None

    bin_preds = np.array(bin_preds)
    five_preds = np.array(five_preds)
    used_sources = np.array(used_sources)

    agree = (bin_preds == five_preds)
    agreement_rate = agree.mean() * 100.0

    print(f"\nAgreement samples: {len(bin_preds)}  (skipped {skipped_unknown} Unknown predictions)")
    print(f"Overall agreement (binary progressive_K5 vs 5-class progressive_K5): {agreement_rate:.2f}%")

    cm = confusion_matrix(five_preds, bin_preds, labels=[0, 1])
    print("\nConfusion matrix (rows=5-class→binary, cols=binary ensemble):")
    print("            non-cancer  cancer")
    print(f"non-cancer  {cm[0][0]:>10d}  {cm[0][1]:>6d}")
    print(f"cancer      {cm[1][0]:>10d}  {cm[1][1]:>6d}")

    # Per-source breakdown (no extra inference — uses collected sources)
    print("\nPer-source agreement:")
    for src in sorted(set(used_sources)):
        mask = used_sources == src
        src_agree = agree[mask].mean() * 100.0
        print(f"  Source {src}: {src_agree:.1f}% agreement ({mask.sum()} samples)")

    return {
        "agreement": agreement_rate,
        "cm": cm,
        "bin_preds": bin_preds,
        "five_preds": five_preds,
    }


print("Running agreement analysis: binary progressive_K5 vs 5-class progressive_K5 on TM...")
print(f"  (using ONNX models from model_registry — no PyTorch architecture needed)\n")
AGREEMENT_TM = agreement_binary_vs_5class_onnx(
    BINARY_PROGRESSIVE_K5_ONNX_DIR,
    FIVECLASS_PROGRESSIVE_K5_ONNX_DIR,
    image_size=CONFIG["image_size"],
)

Binary  ONNX dir: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/onnx
5-class ONNX dir: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_5class/improved_5class_20260209_172821/onnx
Binary models:  ['ensemble_model_1.onnx', 'ensemble_model_2.onnx', 'ensemble_model_3.onnx']
5-class models: ['ensemble_model_1.onnx', 'ensemble_model_2.onnx', 'ensemble_model_3.onnx']
Running agreement analysis: binary progressive_K5 vs 5-class progressive_K5 on TM...
  (using ONNX models from model_registry — no PyTorch architecture needed)

  processed 200/779 images...
  processed 400/779 images...
  processed 600/779 images...

Agreement samples: 777  (skipped 2 Unknown predictions)
Overall agreement (binary progressive_K5 vs 5-class progressive_K5): 80.44%

Confusion matrix (rows=5-class→binary, cols=binary ensemble):
            non-cancer  cancer
non-cancer